# JobCheck — Detector evaluation

Compares the rule-based detector, the ML baseline and their combination. Sections 1–6 use development data only: the EMSCAD dev split (`data/processed/emscad_dev.csv`, from `prepare_emscad.py`) and the Indian dev set (`data/processed/indian_dev.csv`, from `scripts/freeze_indian_split.py`). Section 7 opens the held-out test splits and runs only when `FINAL_EVALUATION` is set to `True`, after rules and thresholds are frozen.

The ML model is loaded from `instance/model.joblib`; train it first with `python -m detector.model`. Metrics are for the suspicious class (label 1).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay, accuracy_score, confusion_matrix,
                             precision_recall_fscore_support)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from detector.model import MODEL_PATH, load, posting_text
from detector.rules import EXPLANATIONS, RULES, THRESHOLD, check
from detector.verdict import MODEL_THRESHOLD, flagged

PROCESSED = PROJECT_ROOT / 'data' / 'processed'

In [ ]:
def load_emscad(path):
    df = pd.read_csv(path, low_memory=False)
    return pd.DataFrame({'text': posting_text(df), 'label': df['fraudulent'].astype(int)})


def load_indian(path):
    df = pd.read_csv(path)
    return pd.DataFrame({'text': df['text'].fillna('').astype(str), 'label': df['label'].astype(int)})


datasets = {'EMSCAD dev': load_emscad(PROCESSED / 'emscad_dev.csv')}
if (PROCESSED / 'indian_dev.csv').exists():
    datasets['Indian dev'] = load_indian(PROCESSED / 'indian_dev.csv')
else:
    print('Indian dev set not found; skipping it.')
for name, df in datasets.items():
    print(name, len(df), 'rows; label counts', df['label'].value_counts().sort_index().to_dict())

## 1. Rule scores

Each posting is scored with and without negation handling.

In [ ]:
def rule_results(df, handle_negation=True):
    results = [check(t, handle_negation=handle_negation) for t in df['text']]
    fired = pd.DataFrame([[rule.id in {h['rule_id'] for h in r['hits']} for rule in RULES] for r in results],
                         columns=[rule.id for rule in RULES], index=df.index)
    return pd.Series([r['score'] for r in results], index=df.index), fired


def metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    return {'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f, 3),
            'accuracy': round(accuracy_score(y_true, y_pred), 3), 'flagged': int(sum(y_pred))}


scores, fired, scores_no_neg = {}, {}, {}
for name, df in datasets.items():
    scores[name], fired[name] = rule_results(df)
    scores_no_neg[name], _ = rule_results(df, handle_negation=False)

## 2. Rule threshold

A posting is flagged when its score is at least the threshold. Recall matters most, but genuine postings should not be routinely flagged. Choose `THRESHOLD` in `detector/rules.py` from the EMSCAD dev rows below.

In [ ]:
name = 'EMSCAD dev'
y = datasets[name]['label']
table = pd.DataFrame([{'threshold': t, **metrics(y, scores[name] >= t)}
                      for t in range(1, int(scores[name].max()) + 2)])
print('Current THRESHOLD =', THRESHOLD)
display(table)

## 3. Rules at the current threshold, and the effect of negation handling

In [ ]:
rows = []
for name, df in datasets.items():
    rows.append({'data': name, 'negation': 'on', **metrics(df['label'], scores[name] >= THRESHOLD)})
    rows.append({'data': name, 'negation': 'off', **metrics(df['label'], scores_no_neg[name] >= THRESHOLD)})
display(pd.DataFrame(rows))

## 4. How often each rule fires

Share of fraudulent and genuine postings in which each rule fires. Rules that fire often on genuine postings are candidates for a lower weight or a narrower pattern.

In [ ]:
for name, df in datasets.items():
    is_fraud = df['label'] == 1
    rates = pd.DataFrame({
        'fraud %': (100 * fired[name][is_fraud].mean()).round(2),
        'genuine %': (100 * fired[name][~is_fraud].mean()).round(2),
        'weight': [rule.weight for rule in RULES],
        'explanation': [EXPLANATIONS[rule.id] for rule in RULES],
    }, index=fired[name].columns).sort_values('fraud %', ascending=False)
    print(name)
    display(rates)
    rates[['fraud %', 'genuine %']][::-1].plot(kind='barh', figsize=(8, 6), color=['#dc3545', '#adb5bd'],
                                                 title=f'Rule hit rate, {name}')
    plt.xlabel('% of postings')
    plt.tight_layout()
    plt.show()

## 5. ML baseline

TF-IDF and logistic regression with class weighting, trained on the text fields of the EMSCAD training split. Choose `MODEL_THRESHOLD` in `detector/verdict.py` from the EMSCAD dev rows below.

In [ ]:
clf = load()
if clf is None:
    raise FileNotFoundError(f'No model at {MODEL_PATH}. Train it first: python -m detector.model')

probs = {name: pd.Series(clf.predict_proba(df['text'])[:, 1], index=df.index) for name, df in datasets.items()}
name = 'EMSCAD dev'
y = datasets[name]['label']
print('Current MODEL_THRESHOLD =', MODEL_THRESHOLD)
display(pd.DataFrame([{'threshold': t, **metrics(y, probs[name] >= t)}
                      for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]]))

fig, ax = plt.subplots(figsize=(5, 4))
for name, df in datasets.items():
    PrecisionRecallDisplay.from_predictions(df['label'], probs[name], name=name, ax=ax)
ax.set_title('ML baseline, precision-recall on dev data')
plt.tight_layout()
plt.show()

## 6. Combined detector

A posting is flagged if either the rule score or the model probability crosses its threshold. The grid below shows the best threshold pairs on the EMSCAD dev split; the table after it compares the three detectors at the current thresholds.

In [ ]:
name = 'EMSCAD dev'
y = datasets[name]['label']
grid = pd.DataFrame([
    {'rule threshold': rt, 'model threshold': mt,
     **metrics(y, [flagged(s, p, rt, mt) for s, p in zip(scores[name], probs[name])])}
    for rt in range(1, int(scores[name].max()) + 2)
    for mt in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
])
display(grid.sort_values(['f1', 'recall'], ascending=False).head(10))

In [ ]:
def predictions(score, prob):
    return {'Rules only': score >= THRESHOLD,
            'ML only': prob >= MODEL_THRESHOLD,
            'Rules + ML': pd.Series([flagged(s, p) for s, p in zip(score, prob)], index=score.index)}


rows = []
for name, df in datasets.items():
    for detector, pred in predictions(scores[name], probs[name]).items():
        rows.append({'data': name, 'detector': detector, **metrics(df['label'], pred)})
display(pd.DataFrame(rows))

fig, axes = plt.subplots(len(datasets), 3, figsize=(12, 3.8 * len(datasets)), squeeze=False)
for row, (name, df) in zip(axes, datasets.items()):
    for ax, (detector, pred) in zip(row, predictions(scores[name], probs[name]).items()):
        cm = confusion_matrix(df['label'], pred, labels=[0, 1])
        ConfusionMatrixDisplay(cm, display_labels=['genuine', 'suspicious']).plot(ax=ax, colorbar=False)
        ax.set_title(f'{detector}, {name}')
plt.tight_layout()
plt.show()

## 7. Final evaluation on the held-out test sets

Run once, after `THRESHOLD`, `MODEL_THRESHOLD` and the rules are frozen. Any change made after seeing these results must be reported separately.

In [ ]:
FINAL_EVALUATION = False

if FINAL_EVALUATION:
    tests = {'EMSCAD test': load_emscad(PROCESSED / 'emscad_test.csv')}
    if (PROCESSED / 'indian_test.csv').exists():
        tests['Indian test'] = load_indian(PROCESSED / 'indian_test.csv')
    rows, errors = [], {}
    for name, df in tests.items():
        score, hits = rule_results(df)
        prob = pd.Series(clf.predict_proba(df['text'])[:, 1], index=df.index)
        preds = predictions(score, prob)
        for detector, pred in preds.items():
            rows.append({'data': name, 'detector': detector, **metrics(df['label'], pred)})
        wrong = preds['Rules + ML'] != (df['label'] == 1)
        errors[name] = pd.DataFrame({
            'error': df['label'][wrong].map({0: 'false positive', 1: 'false negative'}),
            'score': score[wrong],
            'probability': prob[wrong].round(3),
            'rules': hits[wrong].apply(lambda r: ', '.join(r.index[r]), axis=1),
            'excerpt': df['text'][wrong].str[:200],
        })
    display(pd.DataFrame(rows).pivot(index='detector', columns='data', values=['precision', 'recall', 'f1']))

### Error analysis

Every false positive and false negative of the combined detector. Group them by cause (for example: a scam pattern no rule covers, a negation the window missed, a genuine mention of a fee) in the findings.

In [ ]:
if FINAL_EVALUATION:
    for name, e in errors.items():
        print(name, e['error'].value_counts().to_dict())
        display(e)

## 8. Findings

After running the notebook on the full EMSCAD splits and the Indian set, note:

- the chosen rule and model thresholds and why;
- precision, recall and F1 of each detector on each dev set;
- which rules fire often on genuine postings, and what was changed;
- how much negation handling changed the results;
- the final held-out results and the main causes of errors.